# Notebook for preprocessing CTE ocean fluxes

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import os
from global_land_mask import globe



In [2]:
DATA_PATH = "/home/pietaril/Documents/data/CTE_ocean_fluxes/"

filenames = [os.path.join(DATA_PATH,f) for f in os.listdir(DATA_PATH) if f.endswith(".nc")]

ds = xr.open_dataset(filenames[0])
ds



<xarray.Dataset>
Dimensions:    (time: 720, longitude: 250, latitude: 390)
Coordinates:
  * time       (time) datetime64[ns] 2018-09-01 ... 2018-09-30T23:00:00
  * longitude  (longitude) float64 -14.9 -14.7 -14.5 -14.3 ... 34.5 34.7 34.9
  * latitude   (latitude) float64 33.05 33.15 33.25 33.35 ... 71.75 71.85 71.95
Data variables:
    ocean      (time, latitude, longitude) float32 ...
Attributes: (12/19)
    CDI:                        Climate Data Interface version 1.9.9 (https:/...
    Conventions:                CF-1.8
    URL:                        http://carbontracker.wur.nl
    disclaimer:                 This data belongs to the CarbonTracker project
    CDO:                        Climate Data Operators version 1.9.9 (https:/...
    source:                     CTE-HR 1.0. Created using the code from https...
    ...                         ...
    geospatial_lat_resolution:  0.1 degree
    geospatial_lon_resolution:  0.2 degree
    keywords:                   carbon flux
    license:                    CC-BY-4.0
    nominal_resolution:         0.1x0.2 degree
    comment:                    Hourly ocean fluxes, based on a climatology o...

In [ ]:
aoi = [-15, 40, 34, 73]

# Open all input files for the year
datasets = [xr.open_dataset(file) for file in filenames]
#concatenate the files to a single dataset along time dimension
ocn = xr.concat(datasets, dim="time")
ocn = ocn.sortby("time")
#select domain
ocn = ocn.sel(longitude=slice(aoi[0], aoi[1]),
                    latitude=slice(aoi[2], aoi[3]))

In [ ]:
#interpolate to 0.1 x 0.1 deg resolution 
# and check with land-sea mask that ocean fluxes not on land 


def create_lsm(latmin, latmax, lonmin, lonmax):
     """Creates a land-sea mask (land=1, sea=0) in 0.1 deg x 0.1 deg resolution
     for the chosen area.
     """
     lat = np.linspace(latmin,latmax, (latmax-latmin)*10+1)#, dtype=np.float32)
     lon = np.linspace(lonmin,lonmax, (lonmax - lonmin)*10+1)#, dtype=np.float32)
     longrid, latgrid = np.meshgrid(lon,lat)
     lsm = globe.is_land(latgrid, longrid)
   
     return lsm, latgrid, longrid

lsm = create_lsm(aoi[2], aoi[3], aoi[0], aoi[1])



In [10]:
ocn

<xarray.Dataset>
Dimensions:    (time: 8760, longitude: 250, latitude: 380)
Coordinates:
  * time       (time) datetime64[ns] 2018-01-01 ... 2018-12-31T23:00:00
  * longitude  (longitude) float64 -14.9 -14.7 -14.5 -14.3 ... 34.5 34.7 34.9
  * latitude   (latitude) float64 34.05 34.15 34.25 34.35 ... 71.75 71.85 71.95
Data variables:
    ocean      (time, latitude, longitude) float32 -5.065e-08 ... -7.527e-09
Attributes: (12/19)
    CDI:                        Climate Data Interface version 1.9.9 (https:/...
    Conventions:                CF-1.8
    URL:                        http://carbontracker.wur.nl
    disclaimer:                 This data belongs to the CarbonTracker project
    CDO:                        Climate Data Operators version 1.9.9 (https:/...
    source:                     CTE-HR 1.0. Created using the code from https...
    ...                         ...
    geospatial_lat_resolution:  0.1 degree
    geospatial_lon_resolution:  0.2 degree
    keywords:                   carbon flux
    license:                    CC-BY-4.0
    nominal_resolution:         0.1x0.2 degree
    comment:                    Hourly ocean fluxes, based on a climatology o...